# install

In [1]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.7/278.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.4 MB/s eta 0:00:00


In [2]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')


📋 Connecting Google Drive account
to save project settings.



ERROR:vnstock.core.config.ggcolab:Error mounting Drive: mount failed


# Cell 1: Load dữ liệu

In [3]:
### Cell 1: Load dữ liệu (price + volume)
from vnstock import Quote
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

symbols = ["BID", "ACB", "VCB", "VNM", "MSN", "MWG", "HPG", "GAS", "SSI", "VRE"]

raw = {}
for sym in symbols:
    df = Quote(symbol=sym, source="KBS").history(start="2024-01-01", end="2025-12-31", interval="d")
    raw[sym] = df.set_index("time")[["close", "volume"]]

# start từ 2024 để có đủ 252 phiên lookback cho factor momentum tính trong 2025
price_df = pd.DataFrame({sym: d["close"] for sym, d in raw.items()}).sort_index()
volume_df = pd.DataFrame({sym: d["volume"] for sym, d in raw.items()}).sort_index()

price_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-02 07:00:00,34.75,14.80,55.04,57.89,68.4,40.88,18.55,64.82,22.57,22.31
2024-01-03 07:00:00,35.40,15.13,55.70,58.49,68.9,41.60,18.78,65.17,22.88,22.45
2024-01-04 07:00:00,35.28,15.32,56.63,58.49,68.1,41.60,18.75,65.77,23.34,22.60
2024-01-05 07:00:00,35.97,15.41,56.82,58.32,67.9,42.23,18.78,66.19,23.72,22.55
2024-01-08 07:00:00,37.50,15.34,57.22,57.81,66.6,41.60,18.82,65.85,23.68,22.89


# Cell 2: Tính 3 alpha factor

In [4]:
### Cell 2: Tính 3 alpha factor
N_FWD = 5  # forward return horizon (ngày) dùng để đánh giá IC

# 1. Momentum: 12 tháng return, bỏ tháng gần nhất (~21 phiên = 1 tháng, ~252 phiên = 12 tháng)
mom_12_1 = price_df.shift(21) / price_df.shift(252) - 1

# 2. Mean-reversion: z-score của return 5 ngày so với phân phối return 5 ngày trong 20 phiên gần nhất
ret_5d = price_df.pct_change(5)
reversion_z = (ret_5d - ret_5d.rolling(20).mean()) / ret_5d.rolling(20).std()

# 3. Volume factor: volume hôm nay / volume trung bình 20 phiên
volume_ratio = volume_df / volume_df.rolling(20).mean()

factors = {"Momentum_12_1": mom_12_1, "MeanReversion_Z": reversion_z, "VolumeRatio": volume_ratio}
{name: f.iloc[-1].round(3).to_dict() for name, f in factors.items()}  # xem giá trị factor mới nhất

{'Momentum_12_1': {'BID': -0.045,
  'ACB': 0.122,
  'VCB': -0.058,
  'VNM': 0.103,
  'MSN': 0.115,
  'MWG': 0.319,
  'HPG': 0.178,
  'GAS': 0.024,
  'SSI': 0.257,
  'VRE': 1.035},
 'MeanReversion_Z': {'BID': 0.157,
  'ACB': -0.583,
  'VCB': 0.774,
  'VNM': 0.157,
  'MSN': 0.58,
  'MWG': -0.313,
  'HPG': -0.313,
  'GAS': 1.07,
  'SSI': -1.013,
  'VRE': -0.207},
 'VolumeRatio': {'BID': 0.711,
  'ACB': 0.768,
  'VCB': 0.75,
  'VNM': 0.524,
  'MSN': 1.042,
  'MWG': 0.723,
  'HPG': 0.673,
  'GAS': 1.335,
  'SSI': 0.575,
  'VRE': 0.908}}

**- Factor Momentum**

So sánh giá 1 tháng trước với 12 tháng trước => Nếu giá cổ phiếu tăng mạnh trong 1 năm qua thì nó sẽ tăng tiếp

Nhược điểm: Chỉ hoạt động tốt khi thị trường đang trending mạnh, dễ gãy khi thị trường đảo chiều

**- Factor Mean Reversion Z Score**

So sánh return 5 ngày với trung bình 20 ngày và độ lệch chuẩn 20 ngày

Nếu z quá lớn (tăng quá mạnh) => nên bán (vì giá sẽ quay về trung bình)

Hoạt động tốt trong thị trường sideway hoặc mã bluechip


**- Factor Volume Ratio**

Hôm nay có bất thường về thanh khoản so với 20 ngày trước không

Lớn hơn 1: Thanh khoản bùng nổ, dòng tiền vào, cp tăng giá



# Cell 3: IC cho từng factor

In [5]:
### Cell 3: Tính Information Coefficient (IC) cho từng factor
fwd_ret = price_df.shift(-N_FWD) / price_df - 1  # forward return N ngày

def calc_ic(factor_df: pd.DataFrame, fwd_ret_df: pd.DataFrame) -> pd.Series:
    """Cross-sectional Spearman IC giữa factor và forward return, theo từng ngày."""
    ic = {}
    for date in factor_df.index:
        f, r = factor_df.loc[date], fwd_ret_df.loc[date]
        valid = f.notna() & r.notna()
        if valid.sum() >= 5:  # cần đủ số mã để correlation có ý nghĩa
            ic[date] = spearmanr(f[valid], r[valid])[0]
    return pd.Series(ic).dropna()

ic_series = {name: calc_ic(f, fwd_ret) for name, f in factors.items()}

ic_summary = pd.DataFrame({
    name: {
        "Mean IC": s.mean(),
        "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),          # information ratio của IC
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(),
        "N Obs": len(s),
    }
    for name, s in ic_series.items()
}).T.round(3)

ic_summary

,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
Momentum_12_1,-0.045,0.394,-0.115,-1.782,0.459,242.0
MeanReversion_Z,-0.013,0.360,-0.037,-0.806,0.470,470.0
VolumeRatio,-0.014,0.352,-0.040,-0.871,0.491,475.0


Cả 3 factor đều có mean IC < 0, ngược lại với kỳ vọng lý thuyết nhưng t-stat cao, gần với ngưỡng bác bỏ H0 nhất (tức là vẫn có thể có ý nghĩa ở mức nào đó), nhưng lại có ICIR cao nhất

2 factor "Mean Reversion Z" và "Volume ratio" có mean IC ~ 0

Hit rate của cả 3 đều gần với việc đoán ngẫu nhiên

# Cell 4: Plot so sánh

In [6]:
### Cell 4: Plot so sánh 3 factor
PRIMARY_COLORS = ["#5b6fa2", "#2b3d6f", "#fea7e9"]

fig = make_subplots(rows=2, cols=1, subplot_titles=("IC theo thời gian (rolling 20 phiên)", "Mean IC & IC IR"),
                     row_heights=[0.6, 0.4], vertical_spacing=0.15)

for color, (name, s) in zip(PRIMARY_COLORS, ic_series.items()):
    fig.add_trace(go.Scatter(x=s.index, y=s.rolling(20).mean(), name=name,
                              line=dict(color=color, width=2)), row=1, col=1)

fig.add_trace(go.Bar(x=ic_summary.index, y=ic_summary["Mean IC"], name="Mean IC",
                      marker_color=PRIMARY_COLORS, showlegend=False), row=2, col=1)

fig.update_layout(
    title="So sánh Alpha Factors: IC Analysis",
    template="plotly_white", height=700,
    font=dict(size=12), legend=dict(orientation="h", y=1.08),
)
fig.update_yaxes(title_text="IC (rolling 20d)", row=1, col=1)
fig.update_yaxes(title_text="Mean IC", row=2, col=1)
fig.show()

Nguyên nhân kết quả xấu:

Cỡ mẫu 10 mã là quá nhỏ

IC dao động quanh mốc 0, cho thấy độ ổn định của thị trường k kéo dài

N khác nhau giữa các factor

# Cell 5: bổ sung dữ liệu


In [7]:
### Cell 5: Bổ sung dữ liệu open + helper functions kiểu Alpha101
open_df = pd.DataFrame({sym: d["close"] if "open" not in d else d["open"] for sym, d in raw.items()})
# nếu Quote.history() có cột "open", lấy lại từ raw gốc:
open_df = pd.DataFrame({sym: Quote(symbol=sym, source="KBS").history(
    start="2024-01-01", end="2025-12-31", interval="d").set_index("time")["open"] for sym in symbols}).sort_index()

returns = price_df.pct_change()
adv20 = volume_df.rolling(20).mean()

def rank(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-sectional percentile rank (0-1) theo từng ngày."""
    return df.rank(axis=1, pct=True)

def delta(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.diff(d)

def delay(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.shift(d)

def ts_sum(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).sum()

def ts_min(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).min()

def ts_rank(df: pd.DataFrame, d: int) -> pd.DataFrame:
    """Time-series percentile rank của giá trị mới nhất trong window d, theo từng cột."""
    return df.rolling(d).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)

def correlation(df1: pd.DataFrame, df2: pd.DataFrame, d: int) -> pd.DataFrame:
    return df1.rolling(d).corr(df2)

# Cell 6: Các alpha mới

In [8]:
### Cell 6: Alpha A, B, C (WorldQuant-style) + ZLEMA mean-reversion factor

# Alpha A: (-1 * rank(delta(returns, 3))) * correlation(open, volume, 10)
# Ý tưởng: momentum ngắn hạn của return đảo dấu, nhân với mức đồng biến open-volume
alpha_a = (-1 * rank(delta(returns, 3))) * correlation(open_df, volume_df, 10)

# Alpha B: (-1*rank(ts_rank(close,10))) * rank(delta(delta(close,1),1)) * rank(ts_rank(volume/adv20, 5))
# Ý tưởng: kết hợp vị trí giá trong 10 phiên, độ cong (2nd derivative) của giá, và bất thường volume
alpha_b = ((-1 * rank(ts_rank(price_df, 10)))
           * rank(delta(delta(price_df, 1), 1))
           * rank(ts_rank(volume_df / adv20, 5)))

# Alpha C: ternary - nếu xu hướng giá 100 ngày gần như đi ngang (<=5%) -> dùng khoảng cách tới đáy 100 ngày
#          ngược lại (trend rõ) -> dùng momentum ngắn hạn 3 ngày đảo dấu
trend_cond = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)) <= 0.05
alpha_c = pd.DataFrame(
    np.where(trend_cond, -1 * (price_df - ts_min(price_df, 100)), -1 * delta(price_df, 3)),
    index=price_df.index, columns=price_df.columns
)

# ZLEMA mean-reversion factor: giá cao hơn ZLEMA -> kỳ vọng quay về (tín hiệu âm), thấp hơn -> kỳ vọng tăng về (tín hiệu dương)
def zlema(series: pd.Series, period: int = 20) -> pd.Series:
    lag = (period - 1) // 2
    adjusted = series + (series - series.shift(lag))
    return adjusted.ewm(span=period, adjust=False).mean()

zlema_df = price_df.apply(zlema)
alpha_zlema = -(price_df - zlema_df) / zlema_df

new_factors = {"AlphaA_RetCorr": alpha_a, "AlphaB_TSRankVol": alpha_b, "AlphaC_TrendCond": alpha_c, "ZLEMA_Reversion": alpha_zlema}
{name: f.iloc[-1].round(3).to_dict() for name, f in new_factors.items()}

{'AlphaA_RetCorr': {'BID': -0.045,
  'ACB': -0.346,
  'VCB': 0.385,
  'VNM': -0.213,
  'MSN': -0.291,
  'MWG': -0.072,
  'HPG': -0.089,
  'GAS': -0.036,
  'SSI': 0.056,
  'VRE': -0.003},
 'AlphaB_TSRankVol': {'BID': -0.051,
  'ACB': -0.196,
  'VCB': -0.224,
  'VNM': -0.057,
  'MSN': -0.285,
  'MWG': -0.026,
  'HPG': -0.108,
  'GAS': -0.022,
  'SSI': -0.07,
  'VRE': -0.385},
 'AlphaC_TrendCond': {'BID': -0.1,
  'ACB': -0.08,
  'VCB': -0.8,
  'VNM': 0.29,
  'MSN': -1.7,
  'MWG': -1.37,
  'HPG': 0.45,
  'GAS': -16.4,
  'SSI': 0.5,
  'VRE': -1.6},
 'ZLEMA_Reversion': {'BID': -0.001,
  'ACB': 0.002,
  'VCB': -0.006,
  'VNM': 0.011,
  'MSN': -0.011,
  'MWG': -0.004,
  'HPG': 0.008,
  'GAS': -0.013,
  'SSI': 0.02,
  'VRE': -0.008}}

**- Alpha A (ý tưởng từ WorldQuant)**

Return 3 ngày có đảo chiều, đi kèm với volume tăng mạnh => dấu hiệu đảo chiều được xác định bởi dòng tiền

**- Alpha B (ý tưởng từ WorldQuant)**

Giá đang ở đâu, cao hay thấp, có thể quay về mean ko

giá có đang tăng nhanh dần ko

có volume bất thường không

=> tín hiệu: giá cao, đà tăng chậm, volume bất thường = đảo chiều

**- Alpha C (ý tưởng từ WorldQuant)**

đi theo chế độ thị trường

càng gần đáy thì càng nên mua

nếu có trend => dùng mean reversion để tránh đu đỉnh

**- (cá nhân) ZLEMA 20 làm giá chuẩn**

nếu giá vượt => quá mua, nếu giá thấp => quá bán

# Cell 7: IC cho các factor mới

In [9]:
### Cell 7: IC cho các factor mới (dùng lại calc_ic từ Cell 3)
ic_series.update({name: calc_ic(f, fwd_ret) for name, f in new_factors.items()})

ic_summary_full = pd.DataFrame({
    name: {
        "Mean IC": s.mean(),
        "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(),
        "N Obs": len(s),
    }
    for name, s in ic_series.items()
}).T.round(3)

ic_summary_full

,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
Momentum_12_1,-0.045,0.394,-0.115,-1.782,0.459,242.0
MeanReversion_Z,-0.013,0.360,-0.037,-0.806,0.470,470.0
VolumeRatio,-0.014,0.352,-0.040,-0.871,0.491,475.0
AlphaA_RetCorr,0.010,0.332,0.031,0.685,0.515,485.0
AlphaB_TSRankVol,0.008,0.339,0.023,0.496,0.529,471.0
AlphaC_TrendCond,0.064,0.363,0.177,3.926,0.570,491.0
ZLEMA_Reversion,0.005,0.371,0.012,0.268,0.524,485.0


Các chiến lược alpha lần này đã tốt hơn đáng kể, dấu đúng với kỳ vọng hơn

Tuy vậy, chiến lược A, B, ZLEMA có Mean IC rất nhỏ, và t-stat chưa thể bác bỏ H0

Chiến lược đáng kể nhất là chiến lược C, có Mean IC ~ 0.06, t-stat bác bỏ H0, hit rate lên tới 57%, là chiến lược khả quan nhất

# Cell 8: Plot so sánh 7 factor

In [10]:
### Cell 8: Plot so sánh toàn bộ 7 factor
factors.update(new_factors)
COLORS = ["#5b6fa2", "#2b3d6f", "#fea7e9", "#8e99ad", "#7c5cbf", "#c94f7c", "#3fa796"]

fig = make_subplots(rows=2, cols=1, subplot_titles=("Rolling 20d IC theo thời gian", "Mean IC theo factor"),
                     row_heights=[0.6, 0.4], vertical_spacing=0.15)

for color, (name, s) in zip(COLORS, ic_series.items()):
    fig.add_trace(go.Scatter(x=s.index, y=s.rolling(20).mean(), name=name,
                              line=dict(color=color, width=1.8)), row=1, col=1)

fig.add_trace(go.Bar(x=ic_summary_full.index, y=ic_summary_full["Mean IC"],
                      marker_color=COLORS, showlegend=False), row=2, col=1)

fig.update_layout(title="So sánh toàn bộ Alpha Factors: IC Analysis",
                   template="plotly_white", height=750, font=dict(size=11),
                   legend=dict(orientation="h", y=1.1))
fig.update_yaxes(title_text="IC (rolling 20d)", row=1, col=1)
fig.update_yaxes(title_text="Mean IC", row=2, col=1)
fig.show()

Alpha C là chiến lược có hiệu quả nhất, nhưng có vẻ vẫn cần chứng minh thêm về sau, với OOS

Cần kiểm tra xem có phải chỉ chiến thắng các thông số 100 day, 5%, hay là bất kỳ tham số nào
